# Forecast Discounted Method

This notebook will focus on a couple of ideas. The first part of this notebook is to evaluate prediction quality of various models specifically models that extend beyond the OLS framework. All these models’ goal is to predict End of Quarter values. The second part of this notebook is turning that end of quarter values into daily values and testing the exposure and statistical significance of those factors. Currently the main test is to compare the use of MIDAS (Mixed-data Sampling) Regressions relative to OLS. Note that all fundamental values are differenced and therefore it should be assumed so throughout this future. 

# Packages

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from   sklearn.metrics import root_mean_squared_error

import matplotlib.pyplot as plt
from   matplotlib.ticker import FuncFormatter

In [2]:
from tqdm import tqdm
tqdm.pandas()

In [3]:
note_path  = os.getcwd()
repo_path  = os.path.abspath(os.path.join(note_path, ".."))
src_path   = os.path.join(repo_path, "src")
data_path  = os.path.join(repo_path, "data")
db_path    = os.path.join(data_path, "DataBase")
quart_path = os.path.join(db_path, "Quarterlyized")
daily_path = os.path.join(db_path, "DailyData")
model_path = os.path.join(data_path, "Models")
base_path  = os.path.join(model_path, "BaselineModels")

In [4]:
import sys
sys.path.append(src_path)
from OLSModels import OLSModels

In [5]:
sample_tickers = ['SPX Index', 'RTY Index', 'NDX Index']

In [6]:
ticker_path = os.path.join(data_path, "EquityForecastTickerGuide.xlsx")
df_renamer  = pd.read_excel(io = ticker_path, sheet_name = "Renamer")

RENAMER = (df_renamer
    .set_index("old_name")
    .loc[lambda x: x.group == "renamer"]
    .new_name
    .to_dict())

PLOT_RENAMER = (df_renamer
    .set_index("old_name")
    .loc[lambda x: x.group == "plot_renamer"]
    .assign(name = lambda x: x.new_name.str.replace(r"\\n", "\n", regex = True))
    .name
    .to_dict())

FLOAT_FORMAT  = "{:.2f}".format
FLOAT_FORMAT4 = "{:.4f}".format

# Comparing OLS and MIDAS Prediction

At this current version all regressions are ran full-sample and in-sample. We have the following regression we'd like to compare each adding an extra component
1. OLS Model 1: Using lagged fundamental values to predict future fundamental values
2. OLS Model 2: Using lagged fundamentals and retunrs to predict future fundamental values
3. MIDAS Model 1: Using lagged fundamental values and daily returns with the restricted model
4. MIDAS Model 2: Using lagged fundamental values and daily returns with the unrestricted model 

In [7]:
raw_path = r"A:\GitHub\CurrentFundamentalFactorModels\data\Models\BaselineModels"

In [8]:
path = os.path.join(raw_path, "SingleVarPredictiveModels.pkl")
with open(path, "rb") as f: single_models = pickle.load(f)

path = os.path.join(raw_path, "MultiVarPredictiveModels.pkl")
with open(path, "rb") as f: multi_models = pickle.load(f)

In [9]:
def _get_ols_rsquared(models: dict) -> pd.DataFrame:

    df_out = (pd.concat([
        pd.DataFrame.from_dict(
            data   = {
                "model_name": [model_name], 
                "r_squared" : [models[model_name].rsquared],
                "aic"       : [models[model_name].aic]},
            orient = "columns") 
        for model_name in models.keys()]))

    return df_out

In [10]:
df_ols_rsquared = (pd.concat([
    _get_ols_rsquared(single_models).assign(model = "single"),
    _get_ols_rsquared(multi_models).assign(model = "multi")])
    .assign(
        str_split = lambda x: x.model_name.str.split("-"),
        ticker    = lambda x: x.str_split.str[0],
        field     = lambda x: x.str_split.str[1]))

The difference in MIDAS R2 between using vol-targeted returns or not is very small, only adding a small amount, therefore for brevity we'll use volatility adjusted returns

In [25]:
path = r"A:\GitHub\CurrentFundamentalFactorModels\data\Models\MIDASModels\MIDASResults.parquet"
df_aic = (pd
    .read_parquet(path = path, engine = "pyarrow")
    .drop(columns = ["restricted_r2", "unrestricted_r2", "delta_aic"])
    .melt(id_vars = ["field", "ticker", "vol_target"], var_name = "model"))

In [34]:
(df_aic
    .pivot(index = ["field", "ticker", "model"], columns = "vol_target", values = "value")
    .assign(aic_diff = lambda x: x.vol_rtn - x.rtn)
    .reset_index()
    [["field", "model", "aic_diff"]]
    .groupby(["field", "model"])
    .agg("mean")
    .reset_index()
    .pivot(index = "field", columns = "model", values = "aic_diff"))

model,restricted_aic,unrestricted_aic
field,,
CURRENT_EV_TO_T12M_EBITDA,1.431243,12.503589
CURR_ENTP_VAL,-3.833501,14.444088
EBIT,0.140745,-35.306470
EBITDA,-0.993491,-1.033137
EV_TO_T12M_EBITDA,20.157947,29.100873
FREE_CASH_FLOW_PER_SH,0.262108,-3.935521
GROSS_MARGIN,1.957562,7.926136
IS_EPS,0.858731,21.354485


In [21]:
df_ols_aic = (df_ols_rsquared
    [["model", "field", "aic", "model"]])